# Module 4b: AgentCore Evaluations for E-Commerce Multi-Agent System

## Overview

This module sets up **AgentCore Evaluations** to automatically assess agent performance using both **online** and **on-demand** evaluation methods.

**AgentCore Evaluations** provides automated assessment tools that:
- Use LLM-as-a-Judge techniques to evaluate agent responses
- Support 14 built-in evaluators covering quality, safety, and accuracy
- Integrate seamlessly with CloudWatch GenAI Observability

## Evaluation Types

| Type | Description | Use Case |
|------|-------------|----------|
| **Online Evaluation** | Continuously monitors live production traffic | Production monitoring, quality trends |
| **On-Demand Evaluation** | Targeted assessment of specific traces/spans | Issue investigation, testing custom evaluators |

## Built-in Evaluators

| Level | Evaluators |
|-------|------------|
| **Session** | GoalSuccessRate |
| **Trace** | Coherence, Conciseness, ContextRelevance, Correctness, Faithfulness, Harmfulness, Helpfulness, InstructionFollowing, Refusal, ResponseRelevance, Stereotyping |
| **Tool** | ToolParameterAccuracy, ToolSelectionAccuracy |

## What We'll Do

1. **Setup** - Configure prerequisites and IAM permissions
2. **Find Orchestrator Agent** - Retrieve the observability-enabled orchestrator
3. **Online Evaluation** - Create continuous monitoring configuration
4. **Generate Test Data** - Invoke agents to create evaluation data
5. **On-Demand Evaluation** - Evaluate specific traces programmatically
6. **View Results** - Analyze evaluation results in CloudWatch

## Prerequisites

- ✅ Orchestrator agent deployed with observability enabled (from Module 4a)
- ✅ CloudWatch Transaction Search enabled
- ✅ AWS credentials with AgentCore and CloudWatch permissions

**Note:** This notebook retrieves the orchestrator agent directly from AgentCore API - you do not need to run Module 4a in the same session.

### Time: ~25 minutes

## Step 1: Setup and Configuration

In [ ]:
# Install dependencies
!pip install -q boto3 bedrock-agentcore

In [ ]:
import boto3
import json
import time
import uuid
from datetime import datetime, timezone, timedelta
from botocore.exceptions import ClientError

# Get AWS region (can optionally load from 4a, but defaults to session region)
try:
    %store -r REGION
    print(f"✅ Loaded region from previous module: {REGION}")
except:
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'
    print(f"Using session region: {REGION}")

# Initialize AWS clients
sts_client = boto3.client('sts', region_name=REGION)
iam_client = boto3.client('iam', region_name=REGION)
logs_client = boto3.client('logs', region_name=REGION)
agentcore_client = boto3.client('bedrock-agentcore', region_name=REGION)
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

ACCOUNT_ID = sts_client.get_caller_identity()['Account']

print(f"\n📋 Configuration:")
print(f"   Account ID: {ACCOUNT_ID}")
print(f"   Region: {REGION}")

In [ ]:
# Retrieve orchestrator agent directly from AgentCore API
# This allows running this notebook independently without running notebook 4a first

# Note: orchestrator omits "_agent_" due to 48 char name limit
ORCHESTRATOR_AGENT_NAME = 'ecommerce_workshop_orchestrator_observability'

def get_orchestrator_agent() -> dict | None:
    """
    Find the orchestrator agent runtime by name.
    Handles pagination to search through all agents.
    """
    print(f"Searching for orchestrator agent: {ORCHESTRATOR_AGENT_NAME}\n")
    
    try:
        next_token = None
        page_count = 0
        
        while True:
            page_count += 1
            params = {}
            if next_token:
                params['nextToken'] = next_token
            
            response = agentcore_control_client.list_agent_runtimes(**params)
            runtimes = response.get('agentRuntimes', [])
            
            print(f"  Page {page_count}: checking {len(runtimes)} agents...")
            
            for runtime in runtimes:
                runtime_name = runtime.get('agentRuntimeName', '')
                if runtime_name == ORCHESTRATOR_AGENT_NAME:
                    print(f"  ✅ Found orchestrator agent!")
                    return {
                        'name': runtime_name,
                        'id': runtime['agentRuntimeId'],
                        'arn': runtime['agentRuntimeArn'],
                        'status': runtime.get('status', 'UNKNOWN')
                    }
            
            next_token = response.get('nextToken')
            if not next_token:
                break
        
        print(f"  ❌ Orchestrator agent not found after {page_count} page(s)")
        return None
        
    except Exception as e:
        print(f"  ❌ Error searching for agent: {e}")
        return None

ORCHESTRATOR_AGENT = get_orchestrator_agent()

print("\n" + "="*60)
print("ORCHESTRATOR AGENT FOR EVALUATION")
print("="*60)

if ORCHESTRATOR_AGENT:
    print(f"\n✅ ORCHESTRATOR:")
    print(f"   Name: {ORCHESTRATOR_AGENT['name']}")
    print(f"   ID: {ORCHESTRATOR_AGENT['id']}")
    print(f"   ARN: {ORCHESTRATOR_AGENT['arn']}")
    print(f"   Status: {ORCHESTRATOR_AGENT['status']}")
    
    # Create AGENTS dict for compatibility with rest of notebook
    AGENTS = {'orchestrator': ORCHESTRATOR_AGENT}
else:
    print("\n❌ Orchestrator agent not found!")
    print("   Please ensure you have deployed the observability agents from Module 4a.")
    print(f"   Looking for agent named: {ORCHESTRATOR_AGENT_NAME}")
    AGENTS = {}

## Step 2: Create IAM Role for Evaluations

AgentCore Evaluations requires an IAM role with permissions to:
- Invoke Amazon Bedrock models for evaluation
- Read traces from Amazon CloudWatch
- Write evaluation results to Amazon CloudWatch

In [ ]:
EVALUATION_ROLE_NAME = 'ecommerce-workshop-evaluation-role'

def create_evaluation_role() -> str:
    """Create IAM role for AgentCore Evaluations."""
    print(f"Creating evaluation role: {EVALUATION_ROLE_NAME}\n")
    
    # Trust policy for AgentCore Evaluations service
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {
                    "Service": "bedrock-agentcore.amazonaws.com"
                },
                "Action": "sts:AssumeRole",
                "Condition": {
                    "StringEquals": {
                        "aws:SourceAccount": ACCOUNT_ID
                    }
                }
            }
        ]
    }
    
    # Permissions policy
    permissions_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "BedrockModelAccess",
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel",
                    "bedrock:InvokeModelWithResponseStream"
                ],
                "Resource": [
                    f"arn:aws:bedrock:{REGION}::foundation-model/*",
                    f"arn:aws:bedrock:*:{ACCOUNT_ID}:inference-profile/*"
                ]
            },
            {
                "Sid": "CloudWatchLogsRead",
                "Effect": "Allow",
                "Action": [
                    "logs:GetLogEvents",
                    "logs:FilterLogEvents",
                    "logs:StartQuery",
                    "logs:GetQueryResults",
                    "logs:DescribeLogGroups",
                    "logs:DescribeLogStreams"
                ],
                "Resource": [
                    f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:*"
                ]
            },
            {
                "Sid": "CloudWatchLogsWrite",
                "Effect": "Allow",
                "Action": [
                    "logs:CreateLogGroup",
                    "logs:CreateLogStream",
                    "logs:PutLogEvents"
                ],
                "Resource": [
                    f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/bedrock-agentcore/evaluations/*"
                ]
            },
            {
                "Sid": "CloudWatchMetrics",
                "Effect": "Allow",
                "Action": [
                    "cloudwatch:PutMetricData"
                ],
                "Resource": "*",
                "Condition": {
                    "StringEquals": {
                        "cloudwatch:namespace": "Bedrock-AgentCore/Evaluations"
                    }
                }
            },
            {
                "Sid": "XRayRead",
                "Effect": "Allow",
                "Action": [
                    "xray:GetTraceSummaries",
                    "xray:BatchGetTraces",
                    "xray:GetTraceGraph"
                ],
                "Resource": "*"
            }
        ]
    }
    
    try:
        # Check if role exists
        try:
            response = iam_client.get_role(RoleName=EVALUATION_ROLE_NAME)
            role_arn = response['Role']['Arn']
            print(f"ℹ️  Role already exists: {role_arn}")
            
            # Update the permissions policy
            iam_client.put_role_policy(
                RoleName=EVALUATION_ROLE_NAME,
                PolicyName='evaluation-permissions',
                PolicyDocument=json.dumps(permissions_policy)
            )
            print("✅ Updated permissions policy")
            return role_arn
            
        except ClientError as e:
            if e.response['Error']['Code'] != 'NoSuchEntity':
                raise
        
        # Create new role
        response = iam_client.create_role(
            RoleName=EVALUATION_ROLE_NAME,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description='IAM role for AgentCore Evaluations'
        )
        role_arn = response['Role']['Arn']
        print(f"✅ Created role: {role_arn}")
        
        # Attach permissions policy
        iam_client.put_role_policy(
            RoleName=EVALUATION_ROLE_NAME,
            PolicyName='evaluation-permissions',
            PolicyDocument=json.dumps(permissions_policy)
        )
        print("✅ Attached permissions policy")
        
        # Wait for role to propagate
        print("⏳ Waiting for role to propagate...")
        time.sleep(10)
        
        return role_arn
        
    except Exception as e:
        print(f"❌ Error creating role: {e}")
        return None

EVALUATION_ROLE_ARN = create_evaluation_role()
print(f"\n📋 Evaluation Role ARN: {EVALUATION_ROLE_ARN}")

## Step 3: Create Online Evaluation Configuration

Online evaluation continuously monitors agent quality using live production traffic. We'll create an evaluation configuration for the orchestrator agent.

**Configuration includes:**
- Data source: CloudWatch log groups from our observability agents
- Evaluators: Helpfulness, GoalSuccessRate, ToolSelectionAccuracy
- Sampling: 100% (for demo purposes, use lower in production)

In [ ]:
# Define online evaluation configuration
ONLINE_EVAL_CONFIG_NAME = 'ecommerce_workshop_orchestrator_eval'

def get_otel_service_name_from_runtime(runtime_name: str) -> str:
    """
    Derive OTEL service name from runtime name by converting underscores to dashes.
    This ensures the service name matches what 04a sets in OTEL_SERVICE_NAME.
    
    Example: ecommerce_workshop_orchestrator_observability 
           -> ecommerce-workshop-orchestrator-observability
    """
    return runtime_name.replace('_', '-')

def get_agent_log_groups(agent_id: str, endpoint: str = 'DEFAULT') -> list:
    """Get CloudWatch log groups for an agent."""
    return [
        f"/aws/bedrock-agentcore/runtimes/{agent_id}-{endpoint}",
        f"/aws/vendedlogs/bedrock-agentcore/{agent_id}"
    ]

# Get orchestrator agent info
if 'orchestrator' in AGENTS:
    ORCHESTRATOR_INFO = AGENTS['orchestrator']
    ORCHESTRATOR_ID = ORCHESTRATOR_INFO['id']
    ORCHESTRATOR_ARN = ORCHESTRATOR_INFO['arn']
    ORCHESTRATOR_NAME = ORCHESTRATOR_INFO['name']
    
    # Derive service name from runtime name (underscores -> dashes)
    SERVICE_NAME = get_otel_service_name_from_runtime(ORCHESTRATOR_NAME)
    LOG_GROUPS = get_agent_log_groups(ORCHESTRATOR_ID, 'DEFAULT')
    
    print("Orchestrator Agent Configuration:")
    print(f"  Agent ID: {ORCHESTRATOR_ID}")
    print(f"  Agent Runtime Name: {ORCHESTRATOR_NAME}")
    print(f"  OTEL Service Name: {SERVICE_NAME}")
    print(f"  Log Groups:")
    for lg in LOG_GROUPS:
        print(f"    - {lg}")
else:
    print("ERROR: Orchestrator agent not found!")
    ORCHESTRATOR_ID = None
    SERVICE_NAME = None

In [ ]:
def create_online_evaluation_config() -> dict:
    """
    Create online evaluation configuration for continuous agent monitoring.
    
    Uses try-create-then-update pattern. The service monitors CloudWatch log groups
    for traces and applies selected evaluators to score agent performance.
    """
    print("="*60)
    print("CREATING ONLINE EVALUATION CONFIGURATION")
    print("="*60)
    print(f"\nConfig Name: {ONLINE_EVAL_CONFIG_NAME}")
    
    if not ORCHESTRATOR_ID or not SERVICE_NAME:
        print("ERROR: Orchestrator agent not available")
        return None
    
    # Define evaluators to use (max 10)
    evaluators = [
        {"evaluatorId": "Builtin.Helpfulness"},
        {"evaluatorId": "Builtin.GoalSuccessRate"},
        {"evaluatorId": "Builtin.ToolSelectionAccuracy"},
        {"evaluatorId": "Builtin.Coherence"}
    ]
    
    print(f"\nEvaluators:")
    for e in evaluators:
        print(f"  - {e['evaluatorId']}")
    
    # Verify which log groups exist
    existing_log_groups = []
    for lg in LOG_GROUPS:
        try:
            logs_client.describe_log_groups(logGroupNamePrefix=lg)
            existing_log_groups.append(lg)
            print(f"  Found log group: {lg}")
        except:
            print(f"  Log group not found: {lg}")
    
    # Also check aws/spans as a data source
    try:
        logs_client.describe_log_groups(logGroupNamePrefix="aws/spans")
        existing_log_groups.append("aws/spans")
        print(f"  Found log group: aws/spans")
    except:
        pass
    
    if not existing_log_groups:
        print("\nWARNING: No log groups found. Online evaluation may not have data source.")
        existing_log_groups = LOG_GROUPS  # Use configured log groups anyway
    
    print(f"\nService Name: {SERVICE_NAME}")
    print(f"Log Groups: {existing_log_groups[:5]}")  # Max 5 log groups
    
    try:
        # Try to create the configuration
        print("\n  Attempting to create online evaluation config...")
        
        response = agentcore_control_client.create_online_evaluation_config(
            onlineEvaluationConfigName=ONLINE_EVAL_CONFIG_NAME,
            description="Continuous evaluation of e-commerce orchestrator agent",
            rule={
                "samplingConfig": {
                    "samplingPercentage": 100.0  # 100% for demo, use 10-20% in production
                }
            },
            dataSourceConfig={
                "cloudWatchLogs": {
                    "logGroupNames": existing_log_groups[:5],  # Max 5 log groups
                    "serviceNames": [SERVICE_NAME]
                }
            },
            evaluators=evaluators,
            evaluationExecutionRoleArn=EVALUATION_ROLE_ARN,
            enableOnCreate=True
        )
        
        config_id = response.get('onlineEvaluationConfigId')
        print(f"\n  Online evaluation configuration CREATED!")
        print(f"   Config ID: {config_id}")
        print(f"   Status: {response.get('status', 'CREATING')}")
        print(f"   Execution Status: {response.get('executionStatus', 'N/A')}")
        
        return {
            'config_id': config_id,
            'config_name': ONLINE_EVAL_CONFIG_NAME,
            'status': response.get('status'),
            'execution_status': response.get('executionStatus'),
            'action': 'CREATED'
        }
        
    except ClientError as e:
        error_code = e.response['Error']['Code']
        error_msg = e.response['Error']['Message']
        
        if 'ConflictException' in error_code or 'already exists' in error_msg.lower():
            print(f"  Config already exists - retrieving details...")
            
            # Get existing config
            try:
                list_response = agentcore_control_client.list_online_evaluation_configs()
                for config in list_response.get('onlineEvaluationConfigs', []):
                    if config.get('onlineEvaluationConfigName') == ONLINE_EVAL_CONFIG_NAME:
                        config_id = config['onlineEvaluationConfigId']
                        print(f"\n  Found existing online evaluation configuration")
                        print(f"   Config ID: {config_id}")
                        print(f"   Status: {config.get('status', 'UNKNOWN')}")
                        print(f"   Execution Status: {config.get('executionStatus', 'N/A')}")
                        
                        return {
                            'config_id': config_id,
                            'config_name': ONLINE_EVAL_CONFIG_NAME,
                            'status': config.get('status'),
                            'execution_status': config.get('executionStatus'),
                            'action': 'EXISTING'
                        }
            except Exception as list_error:
                print(f"  Error listing configs: {list_error}")
        else:
            print(f"\nError creating config: {error_code}: {error_msg}")
        
        return None
        
    except Exception as e:
        print(f"\nUnexpected error: {e}")
        return None

ONLINE_EVAL_CONFIG = create_online_evaluation_config()

## Step 4: Generate Test Data

Let's invoke the orchestrator agent with various queries to generate trace data for evaluation.

In [ ]:
def invoke_agent(agent_arn: str, prompt: str, session_id: str = None) -> dict:
    """
    Invoke an agent and return the response with session info.
    """
    if not session_id:
        # Generate session ID with min 33 chars (API requirement)
        session_id = f"eval-test-{int(time.time())}-{uuid.uuid4().hex}"
    
    print(f"\nInvoking agent...")
    print(f"  Prompt: {prompt[:60]}..." if len(prompt) > 60 else f"  Prompt: {prompt}")
    print(f"  Session ID: {session_id}")
    
    start_time = time.time()
    
    try:
        payload = json.dumps({"prompt": prompt})
        
        response = agentcore_client.invoke_agent_runtime(
            agentRuntimeArn=agent_arn,
            runtimeSessionId=session_id,
            payload=payload.encode('utf-8'),
            qualifier='DEFAULT'
        )
        
        elapsed = time.time() - start_time
        
        response_body = response.get('response', b'')
        if hasattr(response_body, 'read'):
            response_body = response_body.read()
        
        # Handle bytes
        if isinstance(response_body, bytes):
            response_body = response_body.decode('utf-8')
        
        # Try to parse as JSON, otherwise use as string
        try:
            response_data = json.loads(response_body)
        except (json.JSONDecodeError, TypeError):
            response_data = response_body
        
        # Extract message based on response type
        if isinstance(response_data, dict):
            message = response_data.get('message', str(response_data))
            if isinstance(message, dict):
                message = message.get('message', str(message))
        else:
            message = str(response_data)
        
        print(f"  ✅ Response received in {elapsed:.2f}s")
        print(f"  Response: {str(message)[:100]}..." if len(str(message)) > 100 else f"  Response: {message}")
        
        return {
            'success': True,
            'session_id': session_id,
            'elapsed_time': elapsed,
            'response': response_data,
            'message': message,
            'timestamp': datetime.now(timezone.utc).isoformat()
        }
        
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return {
            'success': False,
            'session_id': session_id,
            'error': str(e),
            'timestamp': datetime.now(timezone.utc).isoformat()
        }

In [ ]:
# Test queries covering different agent domains
TEST_QUERIES = [
    # Order queries
    "What is the status of order ORD-2024-10001?",
    "Track the shipment for order ORD-2024-10002",
    
    # Product queries
    "Find wireless headphones under $100",
    "What are the reviews for product ELEC-001?",
    
    # Account queries  
    "What are the benefits of Gold membership?",
    "How many loyalty points does customer CUST-001 have?",
    
    # Multi-domain queries
    "Check order ORD-2024-10001 status and recommend similar products"
]

print("="*60)
print("GENERATING TEST DATA FOR EVALUATION")
print("="*60)
print(f"\nWill invoke {len(TEST_QUERIES)} queries to generate evaluation data")

INVOCATION_RESULTS = []
SESSION_IDS = []

# Check if orchestrator is available
if 'orchestrator' not in AGENTS:
    print("\n❌ Orchestrator agent not found in AGENTS dict.")
    print("   Please run cell-4 to retrieve the orchestrator agent first.")
else:
    orchestrator_info = AGENTS['orchestrator']
    orchestrator_arn = orchestrator_info['arn']
    orchestrator_status = orchestrator_info.get('status', 'UNKNOWN')
    
    print(f"\nOrchestrator Agent:")
    print(f"  ARN: {orchestrator_arn}")
    print(f"  Status: {orchestrator_status}")
    
    # Allow invocation for READY or ACTIVE status
    if orchestrator_status in ['READY', 'ACTIVE', 'RUNNING']:
        for i, query in enumerate(TEST_QUERIES, 1):
            print(f"\n{'='*60}")
            print(f"Query {i}/{len(TEST_QUERIES)}")
            print(f"{'='*60}")
            
            result = invoke_agent(orchestrator_arn, query)
            INVOCATION_RESULTS.append(result)
            
            # Small delay between invocations
            time.sleep(2)
        
        # Summary
        print("\n" + "="*60)
        print("INVOCATION SUMMARY")
        print("="*60)
        successful = sum(1 for r in INVOCATION_RESULTS if r.get('success'))
        print(f"\n{successful}/{len(INVOCATION_RESULTS)} invocations successful")
        
        # Store session IDs for on-demand evaluation
        SESSION_IDS = [r['session_id'] for r in INVOCATION_RESULTS if r.get('success')]
        print(f"\nSession IDs collected: {len(SESSION_IDS)}")
    else:
        print(f"\n⚠️ Orchestrator agent status is '{orchestrator_status}'")
        print("   Expected: READY, ACTIVE, or RUNNING")
        print("   Please wait for the agent to be ready or check Module 4a deployment.")

## Step 5: On-Demand Evaluation

On-demand evaluation allows us to evaluate specific traces programmatically. This is useful for:
- Testing custom evaluators
- Investigating specific interactions
- Validating fixes for reported issues

### Process:
1. Download span logs from CloudWatch
2. Call the `Evaluate` API with the spans and evaluator

In [ ]:
def query_logs(log_group_name: str, query_string: str, time_range_minutes: int = 60) -> list:
    """Query CloudWatch Logs using Logs Insights."""
    start_time = datetime.now(timezone.utc) - timedelta(minutes=time_range_minutes)
    end_time = datetime.now(timezone.utc)
    
    try:
        query_response = logs_client.start_query(
            logGroupName=log_group_name,
            startTime=int(start_time.timestamp()),
            endTime=int(end_time.timestamp()),
            queryString=query_string
        )
        query_id = query_response['queryId']
        
        # Wait for query to complete
        while True:
            result = logs_client.get_query_results(queryId=query_id)
            if result['status'] in ['Complete', 'Failed']:
                break
            time.sleep(1)
        
        if result['status'] == 'Failed':
            return []
        return result.get('results', [])
    except Exception as e:
        print(f"    Query error for {log_group_name}: {e}")
        return []


def extract_json_messages(query_results: list) -> list:
    """Extract JSON messages from query results."""
    messages = []
    for row in query_results:
        for field in row:
            if field['field'] == '@message':
                value = field['value'].strip()
                if value.startswith('{'):
                    try:
                        messages.append(json.loads(value))
                    except json.JSONDecodeError:
                        pass
    return messages


def is_valid_span(span: dict) -> bool:
    """
    Check if a span has all required fields for on-demand evaluation.
    
    Required fields:
    - scope: dict with 'name' field
    - spanId: string
    - traceId: string
    """
    if not isinstance(span, dict):
        return False
    
    # Must have scope field as a dict
    scope = span.get('scope')
    if not isinstance(scope, dict):
        return False
    
    # scope must have a 'name' field
    if 'name' not in scope:
        return False
    
    # Must have spanId and traceId
    if 'spanId' not in span or 'traceId' not in span:
        return False
    
    return True


def download_session_spans(agent_id: str, session_id: str, time_range_minutes: int = 60) -> list:
    """
    Download span logs from CloudWatch for a specific session.
    
    On-demand evaluation requires spans with:
    - scope.name: identifies the span type
    - spanId: unique span identifier
    - traceId: trace correlation ID
    - attributes.session.id: session correlation
    
    Returns only valid spans that meet all requirements.
    """
    print(f"\nDownloading spans for session: {session_id}")
    
    all_spans = []
    
    # 1. Query aws/spans log group (primary source for OTEL spans)
    # Try multiple possible log group names
    span_log_groups = ["aws/spans", "aws/spans/default"]
    
    for lg in span_log_groups:
        print(f"  Querying {lg}...")
        try:
            # Simple query to get all entries, we'll filter in Python
            query = f"""
            fields @timestamp, @message
            | sort @timestamp asc
            | limit 1000
            """
            results = query_logs(lg, query, time_range_minutes)
            spans = extract_json_messages(results)
            
            # Filter for matching session
            session_spans = []
            for span in spans:
                attrs = span.get('attributes', {})
                span_session = attrs.get('session.id', '')
                if session_id in str(span_session):
                    session_spans.append(span)
            
            print(f"    Found {len(session_spans)} spans matching session")
            all_spans.extend(session_spans)
            
            if session_spans:
                break  # Found spans, no need to try other log groups
        except Exception as e:
            print(f"    Log group {lg} not accessible: {str(e)[:50]}")
    
    # 2. Query agent runtime log group
    agent_log_group = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
    print(f"  Querying {agent_log_group}...")
    try:
        query = f"""
        fields @timestamp, @message
        | sort @timestamp asc
        | limit 1000
        """
        results = query_logs(agent_log_group, query, time_range_minutes)
        spans = extract_json_messages(results)
        
        # Filter for matching session
        session_spans = []
        for span in spans:
            attrs = span.get('attributes', {})
            span_session = attrs.get('session.id', '')
            if session_id in str(span_session):
                session_spans.append(span)
        
        print(f"    Found {len(session_spans)} entries matching session")
        all_spans.extend(session_spans)
    except Exception as e:
        print(f"    Error: {str(e)[:50]}")
    
    # 3. Query vendedlogs log group
    vendedlogs_group = f"/aws/vendedlogs/bedrock-agentcore/{agent_id}"
    print(f"  Querying {vendedlogs_group}...")
    try:
        query = f"""
        fields @timestamp, @message
        | sort @timestamp asc
        | limit 1000
        """
        results = query_logs(vendedlogs_group, query, time_range_minutes)
        spans = extract_json_messages(results)
        
        # Filter for matching session
        session_spans = []
        for span in spans:
            attrs = span.get('attributes', {})
            span_session = attrs.get('session.id', '')
            if session_id in str(span_session):
                session_spans.append(span)
        
        print(f"    Found {len(session_spans)} entries matching session")
        all_spans.extend(session_spans)
    except Exception as e:
        print(f"    Error: {str(e)[:50]}")
    
    # Filter to only include valid spans with all required fields
    valid_spans = [s for s in all_spans if is_valid_span(s)]
    
    print(f"\n  Summary:")
    print(f"    Total entries collected: {len(all_spans)}")
    print(f"    Valid spans (with scope, spanId, traceId): {len(valid_spans)}")
    
    # Debug: Show why spans were filtered out
    if len(all_spans) > len(valid_spans):
        invalid_count = len(all_spans) - len(valid_spans)
        print(f"    Filtered out {invalid_count} invalid entries (missing required fields)")
    
    return valid_spans


def run_on_demand_evaluation(session_spans: list, evaluator_id: str) -> dict:
    """
    Run on-demand evaluation using the Evaluate API.
    
    Args:
        session_spans: List of span objects with required fields (spanId, traceId, scope, etc.)
        evaluator_id: ID of the evaluator (e.g., "Builtin.Helpfulness")
    """
    print(f"\nRunning on-demand evaluation with {evaluator_id}")
    print(f"  Input spans: {len(session_spans)}")
    
    if not session_spans:
        print("  ❌ No valid spans to evaluate")
        return {'evaluator_id': evaluator_id, 'error': 'No valid spans', 'results': []}
    
    # Double-check all spans are valid before sending
    final_spans = [s for s in session_spans if is_valid_span(s)]
    if len(final_spans) != len(session_spans):
        print(f"  ⚠️ Filtered out {len(session_spans) - len(final_spans)} invalid spans")
        print(f"  Final valid spans: {len(final_spans)}")
    
    if not final_spans:
        print("  ❌ No valid spans after final validation")
        return {'evaluator_id': evaluator_id, 'error': 'No valid spans after validation', 'results': []}
    
    try:
        response = agentcore_client.evaluate(
            evaluatorId=evaluator_id,
            evaluationInput={
                "sessionSpans": final_spans
            }
        )
        
        results = response.get('evaluationResults', [])
        print(f"  ✅ Evaluation complete: {len(results)} results")
        
        return {
            'evaluator_id': evaluator_id,
            'results': results,
            'timestamp': datetime.now(timezone.utc).isoformat()
        }
        
    except ClientError as e:
        error_msg = e.response['Error']['Message']
        print(f"  ❌ Evaluation error: {error_msg[:200]}")
        return {
            'evaluator_id': evaluator_id,
            'error': error_msg,
            'results': [],
            'timestamp': datetime.now(timezone.utc).isoformat()
        }
    except Exception as e:
        print(f"  ❌ Unexpected error: {e}")
        return {
            'evaluator_id': evaluator_id,
            'error': str(e),
            'results': [],
            'timestamp': datetime.now(timezone.utc).isoformat()
        }

In [ ]:
# Wait for spans to be available in CloudWatch
print("Waiting 60 seconds for spans to propagate to CloudWatch...")
print("(Spans typically take 1-2 minutes to appear after invocation)")
time.sleep(60)

In [ ]:
# Run on-demand evaluation on collected sessions
print("="*60)
print("ON-DEMAND EVALUATION")
print("="*60)

ON_DEMAND_RESULTS = []

# Evaluators to use for on-demand evaluation
ON_DEMAND_EVALUATORS = [
    "Builtin.Helpfulness",
    "Builtin.Coherence",
    "Builtin.ToolSelectionAccuracy"
]

if SESSION_IDS and ORCHESTRATOR_ID:
    # Use the first successful session for on-demand evaluation
    test_session_id = SESSION_IDS[0]
    
    print(f"\nEvaluating session: {test_session_id}")
    
    # Download spans for this session
    session_spans = download_session_spans(ORCHESTRATOR_ID, test_session_id)
    
    if session_spans:
        # Run evaluation with each evaluator
        for evaluator_id in ON_DEMAND_EVALUATORS:
            print(f"\n{'-'*40}")
            result = run_on_demand_evaluation(session_spans, evaluator_id)
            if result:
                ON_DEMAND_RESULTS.append(result)
            time.sleep(2)  # Small delay between evaluations
    else:
        print("\n⚠️ No spans found for session. Spans may still be propagating.")
        print("   Try running this cell again in a few minutes.")
else:
    print("\n❌ No sessions available for on-demand evaluation.")
    print("   Please run Step 4 to generate test data first.")

In [ ]:
# Display on-demand evaluation results
print("="*60)
print("ON-DEMAND EVALUATION RESULTS")
print("="*60)

if ON_DEMAND_RESULTS:
    for eval_result in ON_DEMAND_RESULTS:
        evaluator_id = eval_result.get('evaluator_id', 'Unknown')
        print(f"\n📊 {evaluator_id}")
        print("-" * 40)
        
        if 'error' in eval_result:
            print(f"   ❌ Error: {eval_result['error']}")
            continue
        
        results = eval_result.get('results', [])
        if not results:
            print("   No results returned")
            continue
        
        for i, result in enumerate(results, 1):
            print(f"\n   Result {i}:")
            print(f"   Score: {result.get('value', 'N/A')}")
            print(f"   Label: {result.get('label', 'N/A')}")
            
            explanation = result.get('explanation', '')
            if explanation:
                # Truncate long explanations
                if len(explanation) > 200:
                    explanation = explanation[:200] + "..."
                print(f"   Explanation: {explanation}")
            
            # Show span context
            context = result.get('context', {}).get('spanContext', {})
            if context:
                print(f"   Session: {context.get('sessionId', 'N/A')[:30]}...")
                if 'traceId' in context:
                    print(f"   Trace: {context.get('traceId', 'N/A')[:20]}...")
            
            # Show token usage
            token_usage = result.get('tokenUsage', {})
            if token_usage:
                print(f"   Tokens: {token_usage.get('inputTokens', 0)} in / {token_usage.get('outputTokens', 0)} out")
else:
    print("\nNo on-demand evaluation results available.")
    print("This could be because:")
    print("  - No spans were found in CloudWatch")
    print("  - The evaluation API encountered an error")
    print("  - Spans are still propagating (wait a few minutes and retry)")

## Step 6: Retrieve and Evaluate a Specific Trace

This demonstrates how to retrieve specific traces from CloudWatch and evaluate them on-demand.

**Note:** AgentCore traces are delivered to CloudWatch GenAI Observability via log delivery (to the `aws/spans` log group), not traditional X-Ray trace storage. The `get_trace_summaries` X-Ray API won't find these traces.

In [ ]:
# Query traces from CloudWatch aws/spans log group (not traditional X-Ray)
# AgentCore traces are delivered to CloudWatch GenAI Observability via log delivery

def get_recent_traces_from_cloudwatch(time_range_minutes: int = 30, limit: int = 10) -> list:
    """
    Get recent traces from CloudWatch aws/spans log group.
    
    Note: AgentCore traces are delivered to CloudWatch GenAI Observability
    via log delivery, not traditional X-Ray trace storage.
    """
    print(f"\nRetrieving recent traces from CloudWatch aws/spans...")
    
    end_time = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_time = int((datetime.now(timezone.utc) - timedelta(minutes=time_range_minutes)).timestamp() * 1000)
    
    traces_found = []
    trace_ids_seen = set()
    
    spans_log_groups = ["aws/spans", "aws/spans/default"]
    
    for log_group in spans_log_groups:
        try:
            # Get recent log streams
            streams_response = logs_client.describe_log_streams(
                logGroupName=log_group,
                orderBy='LastEventTime',
                descending=True,
                limit=10
            )
            
            streams = streams_response.get('logStreams', [])
            if not streams:
                continue
            
            print(f"  Found {len(streams)} log streams in {log_group}")
            
            for stream in streams:
                stream_name = stream['logStreamName']
                
                events_response = logs_client.get_log_events(
                    logGroupName=log_group,
                    logStreamName=stream_name,
                    startTime=start_time,
                    endTime=end_time,
                    limit=200
                )
                
                for event in events_response.get('events', []):
                    try:
                        span_data = json.loads(event.get('message', ''))
                        trace_id = span_data.get('traceId', '')
                        
                        if trace_id and trace_id not in trace_ids_seen:
                            trace_ids_seen.add(trace_id)
                            
                            # Extract service name from scope
                            scope = span_data.get('scope', {})
                            service_name = scope.get('name', 'unknown')
                            
                            # Get span name
                            span_name = span_data.get('name', 'unknown')
                            
                            traces_found.append({
                                'trace_id': trace_id,
                                'span_name': span_name,
                                'service': service_name,
                                'timestamp': event.get('timestamp', 0)
                            })
                            
                            if len(traces_found) >= limit:
                                break
                    except json.JSONDecodeError:
                        pass
                
                if len(traces_found) >= limit:
                    break
            
            if traces_found:
                print(f"  ✅ Found {len(traces_found)} unique traces in {log_group}")
                break
                
        except logs_client.exceptions.ResourceNotFoundException:
            print(f"  Log group {log_group} not found")
        except Exception as e:
            print(f"  Error querying {log_group}: {str(e)[:50]}")
    
    return traces_found


# Get recent traces from CloudWatch
RECENT_TRACES = get_recent_traces_from_cloudwatch(time_range_minutes=30)

if RECENT_TRACES:
    print("\n" + "="*60)
    print("RECENT AGENTCORE TRACES (from CloudWatch aws/spans)")
    print("="*60)
    
    for i, trace in enumerate(RECENT_TRACES[:5], 1):
        print(f"\n{i}. Trace ID: {trace['trace_id']}")
        print(f"   Span: {trace['span_name']}")
        print(f"   Service: {trace['service']}")
else:
    print("\n⚠️ No traces found in aws/spans log group.")
    print("   Traces may still be propagating.")
    print("   You can still proceed with evaluation using session_spans from Step 5.")

In [ ]:
# Evaluate a specific trace using trace ID from the spans
print("="*60)
print("TRACE-SPECIFIC EVALUATION")
print("="*60)

TRACE_EVAL_RESULT = None

if session_spans:
    # Extract unique trace IDs from the actual spans we have
    trace_ids_in_spans = set()
    for span in session_spans:
        trace_id = span.get('traceId')
        if trace_id:
            trace_ids_in_spans.add(trace_id)
    
    print(f"\nFound {len(trace_ids_in_spans)} unique trace ID(s) in session spans:")
    for tid in list(trace_ids_in_spans)[:5]:
        print(f"  - {tid}")
    
    if trace_ids_in_spans:
        # Use the first trace ID from the spans
        target_trace_id = list(trace_ids_in_spans)[0]
        
        print(f"\nEvaluating trace: {target_trace_id}")
        
        try:
            # Run evaluation targeting this specific trace
            response = agentcore_client.evaluate(
                evaluatorId="Builtin.Helpfulness",
                evaluationInput={
                    "sessionSpans": session_spans
                },
                evaluationTarget={
                    "traceIds": [target_trace_id]
                }
            )
            
            results = response.get('evaluationResults', [])
            
            if results:
                result = results[0]
                print(f"\n✅ Trace Evaluation Complete")
                print(f"   Evaluator: Builtin.Helpfulness")
                print(f"   Score: {result.get('value', 'N/A')}")
                print(f"   Label: {result.get('label', 'N/A')}")
                
                explanation = result.get('explanation', '')
                if explanation:
                    print(f"   Explanation: {explanation[:300]}..." if len(explanation) > 300 else f"   Explanation: {explanation}")
                
                TRACE_EVAL_RESULT = result
            else:
                print("\n⚠️ No evaluation results returned for this trace")
                
        except Exception as e:
            print(f"\n❌ Trace evaluation error: {e}")
    else:
        print("\n⚠️ No trace IDs found in session spans")
else:
    print("\n⚠️ No session spans available")
    print("   Please run cell-14 and cell-16 first to download spans")

## Step 7: View Evaluation Results in CloudWatch

Evaluation results are stored in CloudWatch and can be viewed in:
1. **GenAI Observability Dashboard** - Aggregated scores and trends
2. **CloudWatch Metrics** - Evaluation score metrics
3. **CloudWatch Logs** - Detailed evaluation results

In [ ]:
def check_evaluation_log_group() -> str:
    """
    Check if evaluation results log group exists.
    """
    if not ONLINE_EVAL_CONFIG:
        return None
    
    config_id = ONLINE_EVAL_CONFIG.get('config_id')
    log_group = f"/aws/bedrock-agentcore/evaluations/results/{config_id}"
    
    try:
        logs_client.describe_log_groups(logGroupNamePrefix=log_group)
        return log_group
    except:
        return None


def query_evaluation_results(log_group: str, limit: int = 20) -> list:
    """
    Query evaluation results from CloudWatch Logs.
    """
    print(f"\nQuerying evaluation results from: {log_group}")
    
    try:
        query = """
        fields @timestamp, @message
        | filter @message like /evaluation/
        | sort @timestamp desc
        | limit 20
        """
        
        end_time = datetime.now(timezone.utc)
        start_time = end_time - timedelta(hours=1)
        
        query_response = logs_client.start_query(
            logGroupName=log_group,
            startTime=int(start_time.timestamp()),
            endTime=int(end_time.timestamp()),
            queryString=query
        )
        
        query_id = query_response['queryId']
        
        while True:
            result = logs_client.get_query_results(queryId=query_id)
            if result['status'] in ['Complete', 'Failed']:
                break
            time.sleep(1)
        
        if result['status'] == 'Complete':
            return result.get('results', [])
        return []
        
    except Exception as e:
        print(f"  Error querying logs: {e}")
        return []


# Check evaluation results
print("="*60)
print("CLOUDWATCH EVALUATION RESULTS")
print("="*60)

eval_log_group = check_evaluation_log_group()

if eval_log_group:
    print(f"\n✅ Evaluation log group found: {eval_log_group}")
    
    # Query for results
    eval_logs = query_evaluation_results(eval_log_group)
    
    if eval_logs:
        print(f"\nFound {len(eval_logs)} evaluation result entries")
    else:
        print("\nNo evaluation results yet. Online evaluation results will appear")
        print("after the evaluation job processes agent traces.")
else:
    print("\n⚠️ Evaluation log group not found yet.")
    print("   Online evaluation results will be created after the first evaluation runs.")

In [ ]:
# Print CloudWatch console links
print("\n" + "="*70)
print("CLOUDWATCH CONSOLE LINKS")
print("="*70)

print(f"\n🔍 GenAI Observability Dashboard (with Evaluations):")
print(f"   https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#genai-observability:bedrockAgentCore")

print(f"\n📊 CloudWatch Metrics (Evaluation Scores):")
print(f"   https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#metricsV2:graph=~();namespace=Bedrock-AgentCore/Evaluations")

if ONLINE_EVAL_CONFIG:
    config_id = ONLINE_EVAL_CONFIG.get('config_id', '')
    print(f"\n📝 Evaluation Results Log Group:")
    print(f"   /aws/bedrock-agentcore/evaluations/results/{config_id}")

print(f"\n📈 X-Ray Traces:")
print(f"   https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#xray:traces")

## Step 8: Summary and Cleanup Options

In [ ]:
print("\n" + "="*70)
print("MODULE 4B COMPLETE: AGENTCORE EVALUATIONS")
print("="*70)

print("\n📋 WHAT WE ACCOMPLISHED")
print("-"*70)

print("\n1. SETUP")
print(f"   ✅ Created evaluation IAM role: {EVALUATION_ROLE_NAME}")
print(f"   ✅ Retrieved orchestrator agent from AgentCore API")

print("\n2. ONLINE EVALUATION")
if ONLINE_EVAL_CONFIG:
    print(f"   ✅ Config: {ONLINE_EVAL_CONFIG.get('config_name')}")
    print(f"   ✅ Config ID: {ONLINE_EVAL_CONFIG.get('config_id')}")
    print(f"   ✅ Status: {ONLINE_EVAL_CONFIG.get('status')}")
    print(f"   ✅ Evaluators: Helpfulness, GoalSuccessRate, ToolSelectionAccuracy, Coherence")
else:
    print("   ⚠️ Online evaluation not configured")

print("\n3. TEST INVOCATIONS")
successful_invocations = sum(1 for r in INVOCATION_RESULTS if r.get('success'))
print(f"   ✅ Invocations: {successful_invocations}/{len(INVOCATION_RESULTS)} successful")
print(f"   ✅ Sessions collected: {len(SESSION_IDS)}")

print("\n4. ON-DEMAND EVALUATION")
if ON_DEMAND_RESULTS:
    print(f"   ✅ Evaluations run: {len(ON_DEMAND_RESULTS)}")
    for result in ON_DEMAND_RESULTS:
        if 'error' not in result:
            num_results = len(result.get('results', []))
            print(f"      - {result['evaluator_id']}: {num_results} results")
else:
    print("   ⚠️ No on-demand evaluations completed")

print("\n" + "="*70)
print("KEY TAKEAWAYS")
print("="*70)
print("""
1. Online Evaluation:
   - Continuously monitors agent quality with live traffic
   - Results automatically stored in CloudWatch
   - Scores published as CloudWatch metrics

2. On-Demand Evaluation:
   - Download spans from CloudWatch
   - Call Evaluate API with specific evaluator
   - Can target specific traces or spans

3. Built-in Evaluators:
   - Session-level: GoalSuccessRate
   - Trace-level: Helpfulness, Coherence, Correctness, etc.
   - Tool-level: ToolSelectionAccuracy, ToolParameterAccuracy

4. Integration with Observability:
   - View evaluations in GenAI Observability dashboard
   - Correlate evaluation scores with traces and logs
   - Set alarms on evaluation metrics
""")
print("="*70)

In [ ]:
# Store variables for use in other modules
%store ONLINE_EVAL_CONFIG
%store EVALUATION_ROLE_ARN
%store SESSION_IDS
%store ON_DEMAND_RESULTS

print("\n✅ Variables stored for use in subsequent modules")

In [ ]:
# Optional: Cleanup function
def cleanup_evaluation_resources(delete_config: bool = False, delete_role: bool = False):
    """
    Clean up evaluation resources.
    
    WARNING: This will delete the online evaluation configuration and/or IAM role.
    """
    print("\n⚠️ CLEANUP - This will delete resources!")
    
    if delete_config and ONLINE_EVAL_CONFIG:
        config_id = ONLINE_EVAL_CONFIG.get('config_id')
        print(f"\nDeleting online evaluation config: {config_id}")
        try:
            agentcore_control_client.delete_online_evaluation_config(
                onlineEvaluationConfigId=config_id
            )
            print("✅ Config deleted")
        except Exception as e:
            print(f"❌ Error: {e}")
    
    if delete_role:
        print(f"\nDeleting IAM role: {EVALUATION_ROLE_NAME}")
        try:
            # Delete inline policies first
            policies = iam_client.list_role_policies(RoleName=EVALUATION_ROLE_NAME)
            for policy_name in policies.get('PolicyNames', []):
                iam_client.delete_role_policy(
                    RoleName=EVALUATION_ROLE_NAME,
                    PolicyName=policy_name
                )
            
            iam_client.delete_role(RoleName=EVALUATION_ROLE_NAME)
            print("✅ Role deleted")
        except Exception as e:
            print(f"❌ Error: {e}")

# Uncomment to run cleanup:
# cleanup_evaluation_resources(delete_config=True, delete_role=True)